<a href="https://colab.research.google.com/github/julioo2810/Curso_SENAC_IA/blob/main/Exercicios_regressao_entregas_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lista de Exercícios – Regressão Linear com Base de Logística

## Problema: prever tempo de entrega

Neste notebook, vamos trabalhar com uma base simulada de uma empresa de entregas.

A empresa quer prever o **tempo de entrega em minutos** com base em características do pedido e da entrega.

## Objetivo geral

Criar modelos de regressão linear para prever:

> quanto tempo uma entrega deve demorar.

Essa é uma aplicação mais próxima de problemas reais de empresas, e-commerce, logística e sistemas de roteirização.

# Contexto da base

Cada linha representa um pedido entregue.

## Colunas

| Coluna | Descrição |
|---|---|
| id_pedido | Identificador do pedido |
| distancia_km | Distância entre loja/centro de distribuição e cliente |
| peso_kg | Peso total do pedido |
| qtd_itens | Quantidade de itens no pedido |
| valor_pedido | Valor total do pedido |
| trafego | Condição do tráfego: baixo, medio ou alto |
| clima | Condição climática: normal, chuva ou forte_chuva |
| tipo_entrega | normal ou expressa |
| regiao | centro, bairro ou zona_rural |
| tempo_entrega_min | Tempo de entrega em minutos |
| atrasou | Indica se a entrega passou de 120 minutos |

## Variável alvo para regressão

A variável que queremos prever é:

`tempo_entrega_min`

# Importação das bibliotecas

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# Criando a base de dados

In [7]:

np.random.seed(42)

n = 200

df = pd.DataFrame({
    'id_pedido': range(1, n+1),
    'distancia_km': np.random.uniform(1, 20, n),
    'peso_kg': np.random.uniform(0.5, 10, n),
    'qtd_itens': np.random.randint(1, 10, n),
    'valor_pedido': np.random.uniform(20, 500, n),
    'trafego': np.random.choice(['baixo', 'medio', 'alto'], n),
    'clima': np.random.choice(['normal', 'chuva', 'forte_chuva'], n),
    'tipo_entrega': np.random.choice(['normal', 'expressa'], n),
    'regiao': np.random.choice(['centro', 'bairro', 'zona_rural'], n)
})

df['tempo_entrega_min'] = (
    df['distancia_km'] * 4 +
    df['peso_kg'] * 2 +
    np.where(df['trafego'] == 'alto', 30,
             np.where(df['trafego'] == 'medio', 15, 5)) +
    np.where(df['tipo_entrega'] == 'expressa', -10, 0) +
    np.random.normal(0, 5, n)
)

df['atrasou'] = df['tempo_entrega_min'] > 120

df.head()

,id_pedido,distancia_km,peso_kg,qtd_itens,valor_pedido,trafego,clima,tipo_entrega,regiao,tempo_entrega_min,atrasou
0,1,8.116262,6.599301,9,143.957306,alto,forte_chuva,normal,centro,80.362891,False
1,2,19.063572,1.299330,1,240.385163,alto,forte_chuva,normal,centro,113.083014,False
2,3,14.907885,2.035473,1,490.415636,alto,forte_chuva,expressa,centro,82.085704,False
3,4,12.374511,9.036265,4,256.456685,medio,normal,expressa,bairro,72.808592,False
4,5,3.964354,6.261076,9,177.800773,alto,normal,normal,centro,51.017027,False


# Parte A – Exploração inicial da base

## Exercício 1
Mostre as 10 primeiras linhas da base.

## Exercício 2
Mostre as 10 últimas linhas da base.

## Exercício 3
Use `.info()` para verificar os tipos de dados.

## Exercício 4
Use `.describe()` para analisar as variáveis numéricas.

## Exercício 5
Verifique se há valores nulos.

## Exercício 6
Quantas linhas e colunas existem?

## Exercício 7
Qual é a variável alvo do problema de regressão?

## Exercício 8
Quais variáveis parecem ser boas candidatas para prever tempo de entrega?

## Exercício 9
Quais variáveis são categóricas?

## Exercício 10
Por que não podemos usar variáveis categóricas diretamente na regressão linear sem tratamento?

In [5]:
# Espaço para resolver a Parte A
#1
df.head(10)

,id_pedido,distancia_km,peso_kg,qtd_itens,valor_pedido,trafego,clima,tipo_entrega,regiao,tempo_entrega_min,atrasou
0,1,8.116262,6.599301,9,143.957306,alto,forte_chuva,normal,centro,80.362891,False
1,2,19.063572,1.299330,1,240.385163,alto,forte_chuva,normal,centro,113.083014,False
2,3,14.907885,2.035473,1,490.415636,alto,forte_chuva,expressa,centro,82.085704,False
3,4,12.374511,9.036265,4,256.456685,medio,normal,expressa,bairro,72.808592,False
4,5,3.964354,6.261076,9,177.800773,alto,normal,normal,centro,51.017027,False
5,6,3.963896,0.587372,6,324.032410,alto,chuva,normal,bairro,46.581849,False
6,7,2.103589,1.463980,3,135.269897,baixo,chuva,expressa,centro,12.259385,False
7,8,17.457347,6.803267,1,56.414397,alto,normal,normal,zona_rural,110.132384,False
8,9,12.421185,0.548085,4,81.862267,medio,chuva,normal,bairro,70.896617,False
9,10,14.453379,2.027676,9,81.462003,medio,normal,expressa,zona_rural,67.326798,False


In [8]:
#2
df.tail(10)

,id_pedido,distancia_km,peso_kg,qtd_itens,valor_pedido,trafego,clima,tipo_entrega,regiao,tempo_entrega_min,atrasou
190,191,2.768953,9.909799,1,414.203506,baixo,forte_chuva,normal,bairro,44.017318,False
191,192,18.047099,4.419868,8,73.103234,medio,chuva,expressa,bairro,87.653802,False
192,193,18.107943,4.034172,7,426.297100,medio,forte_chuva,expressa,bairro,86.397200,False
193,194,13.028928,7.875923,3,81.194558,baixo,normal,expressa,zona_rural,71.971459,False
194,195,7.441566,3.737634,1,210.697899,alto,chuva,expressa,zona_rural,55.888532,False
195,196,7.634982,9.342195,5,402.701776,baixo,forte_chuva,normal,bairro,60.533419,False
196,197,14.793158,8.654921,4,91.960365,alto,forte_chuva,expressa,zona_rural,97.129009,False
197,198,18.045095,4.575443,8,130.040670,medio,normal,expressa,centro,88.231504,False
198,199,17.854642,7.633275,1,366.681233,alto,chuva,normal,centro,110.598594,False
199,200,15.817635,7.668157,1,365.617538,medio,forte_chuva,expressa,centro,81.196622,False


In [9]:
#3
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_pedido          200 non-null    int64  
 1   distancia_km       200 non-null    float64
 2   peso_kg            200 non-null    float64
 3   qtd_itens          200 non-null    int64  
 4   valor_pedido       200 non-null    float64
 5   trafego            200 non-null    object 
 6   clima              200 non-null    object 
 7   tipo_entrega       200 non-null    object 
 8   regiao             200 non-null    object 
 9   tempo_entrega_min  200 non-null    float64
 10  atrasou            200 non-null    bool   
dtypes: bool(1), float64(4), int64(2), object(4)
memory usage: 15.9+ KB


In [12]:
#4
df.describe(include='all')

,id_pedido,distancia_km,peso_kg,qtd_itens,valor_pedido,trafego,clima,tipo_entrega,regiao,tempo_entrega_min,atrasou
count,200.000000,200.000000,200.000000,200.000000,200.000000,200,200,200,200,200.000000,200
unique,NaN,NaN,NaN,NaN,NaN,3,3,2,3,NaN,2
top,NaN,NaN,NaN,NaN,NaN,alto,chuva,normal,centro,NaN,False
freq,NaN,NaN,NaN,NaN,NaN,75,69,106,69,NaN,198
mean,100.500000,10.196118,5.291564,4.755000,239.743589,NaN,NaN,NaN,NaN,64.843055,NaN
std,57.879185,5.602937,2.783511,2.601695,134.015794,NaN,NaN,NaN,NaN,27.575055,NaN
min,1.000000,1.104920,0.548085,1.000000,29.634175,NaN,NaN,NaN,NaN,2.968569,NaN
25%,50.750000,5.343066,2.983894,2.750000,115.660517,NaN,NaN,NaN,NaN,47.122497,NaN
50%,100.500000,10.395239,5.645579,5.000000,225.703036,NaN,NaN,NaN,NaN,64.319372,NaN
75%,150.250000,15.380333,7.550893,7.000000,354.144089,NaN,NaN,NaN,NaN,84.028265,NaN


In [15]:
#5
df.isnull().values.any()

np.False_

In [18]:
#6
df.shape

(200, 11)

In [ ]:
#7
#tempo_entrega_min

In [ ]:
#8
#As melhores pra prever o tempo são distancia_km, trafego e tipo_entrega, porque impactam direto na entrega.


In [ ]:
#9
#As variáveis categóricas são: **trafego, clima, tipo_entrega e regiao**.


In [ ]:
#10
#Porque são só rótulos e não números de verdade. A regressão faz contas e vai interpretar errado. Por isso precisa transformar em números antes como one hot encoding


# Parte B – Estatística descritiva

## Exercício 11
Calcule a média do tempo de entrega.

## Exercício 12
Calcule a mediana do tempo de entrega.

## Exercício 13
Calcule o desvio padrão do tempo de entrega.

## Exercício 14
Calcule o tempo mínimo e máximo de entrega.

## Exercício 15
Calcule os quartis de `tempo_entrega_min`.

## Exercício 16
Interprete a diferença entre média e mediana.

## Exercício 17
Existe muita variação nos tempos de entrega? Use o desvio padrão para justificar.

## Exercício 18
Existe algum possível outlier no tempo de entrega?

## Exercício 19
Compare o tempo médio de entrega por tipo de entrega.

## Exercício 20
Compare o tempo médio de entrega por condição de tráfego.

In [20]:
# Espaço para resolver a Parte B
#11
round(df['tempo_entrega_min'].mean(), 2)


np.float64(64.84)

In [21]:
#12
round(df['tempo_entrega_min'].median(), 2)


64.32

In [22]:
#13
round(df['tempo_entrega_min'].std(), 2)


27.58

In [23]:
#14
round(df['tempo_entrega_min'].min(), 2), round(df['tempo_entrega_min'].max(), 2)

(2.97, 121.51)

In [24]:
#15
df['tempo_entrega_min'].quantile([0.25, 0.5, 0.75]).round(2)

,tempo_entrega_min
0.25,47.12
0.50,64.32
0.75,84.03


In [ ]:
#16
#A média e a mediana estarem parecidas indica que os dados estão bem equilibrados, sem valores extremos puxando o resultado.


In [ ]:
#17
#Existe uma variação relativamente alta nos tempos de entrega, pois o desvio padrão de 27 minutos é significativo em relação à média, indicando que os valores estão bem dispersos.

In [ ]:
#18
#Sim, podem existir outliers, pois há valores de tempo de entrega bem acima da média, o que indica possíveis entregas fora do padrão

In [25]:
#19
df.groupby('tipo_entrega')['tempo_entrega_min'].mean().round(2)
#O tempo médio de entrega expressa é menor (61,10 min) em comparação com a entrega normal (68,16 min).



,tempo_entrega_min
tipo_entrega,
expressa,61.10
normal,68.16


In [26]:
#20
df.groupby('trafego')['tempo_entrega_min'].mean().round(2)
#O tempo médio de entrega é maior em tráfego alto (80min), intermediário em tráfego médio (63min) e menor em tráfego baixo (46min), o trânsito impacta fortemente no tempo de entrega.

,tempo_entrega_min
trafego,
alto,80.77
baixo,46.70
medio,63.74


# Parte C – Visualização

## Exercício 21
Faça um histograma de `tempo_entrega_min`.

## Exercício 22
Faça um boxplot de `tempo_entrega_min`.

## Exercício 23
Faça um gráfico de dispersão entre `distancia_km` e `tempo_entrega_min`.

## Exercício 24
Faça um gráfico de dispersão entre `peso_kg` e `tempo_entrega_min`.

## Exercício 25
Faça um gráfico de dispersão entre `qtd_itens` e `tempo_entrega_min`.

## Exercício 26
Faça um boxplot de tempo de entrega por `trafego`.

## Exercício 27
Faça um boxplot de tempo de entrega por `clima`.

## Exercício 28
Faça um boxplot de tempo de entrega por `tipo_entrega`.

## Exercício 29
Visualmente, qual variável parece mais relacionada ao tempo de entrega?

## Exercício 30
O gráfico sugere relação linear entre distância e tempo?

In [ ]:
# Espaço para resolver a Parte C

# Parte D – Correlação

Para calcular correlação, use apenas variáveis numéricas.

## Exercício 31
Crie um DataFrame apenas com variáveis numéricas.

## Exercício 32
Calcule a matriz de correlação.

## Exercício 33
Faça um heatmap da matriz de correlação.

## Exercício 34
Qual variável numérica tem maior correlação positiva com `tempo_entrega_min`?

## Exercício 35
Qual variável numérica parece ter menor relação com `tempo_entrega_min`?

## Exercício 36
Correlação alta prova causalidade? Explique.

## Exercício 37
Se `valor_pedido` tiver baixa correlação com tempo, isso significa que nunca deve ser usado? Explique.

## Exercício 38
A variável `distancia_km` parece importante? Justifique.

## Exercício 39
A variável `peso_kg` parece importante? Justifique.

## Exercício 40
Quais variáveis você escolheria para um primeiro modelo simples?

In [ ]:
# Espaço para resolver a Parte D

# Parte E – Regressão linear simples

Vamos começar com um modelo simples:

`distancia_km` → `tempo_entrega_min`

## Exercício 41
Crie `X` com a coluna `distancia_km`.

## Exercício 42
Crie `y` com `tempo_entrega_min`.

## Exercício 43
Treine um modelo de regressão linear.

## Exercício 44
Mostre o coeficiente angular.

## Exercício 45
Mostre o intercepto.

## Exercício 46
Interprete o coeficiente angular.

## Exercício 47
Faça previsões para todos os pedidos.

## Exercício 48
Crie uma tabela com tempo real, tempo previsto e erro.

## Exercício 49
Calcule MAE, RMSE e R².

## Exercício 50
Interprete as métricas.

In [ ]:
# Espaço para resolver a Parte E

# Parte F – Visualizando a reta do modelo simples

## Exercício 51
Faça um gráfico de dispersão entre distância e tempo real.

## Exercício 52
Adicione a reta de regressão no gráfico.

## Exercício 53
O modelo parece representar bem os dados?

## Exercício 54
Existem pontos muito distantes da reta?

## Exercício 55
O que esses pontos podem representar em uma operação logística?

In [ ]:
# Espaço para resolver a Parte F

# Parte G – Fazendo previsões reais

Use o modelo simples com distância.

## Exercício 56
Preveja o tempo de entrega para uma entrega de 5 km.

## Exercício 57
Preveja o tempo de entrega para uma entrega de 20 km.

## Exercício 58
Preveja o tempo de entrega para uma entrega de 60 km.

## Exercício 59
As previsões fazem sentido?

## Exercício 60
Quais informações importantes o modelo simples está ignorando?

In [ ]:
# Espaço para resolver a Parte G

# Parte H – Tratamento de variáveis categóricas

Para usar variáveis como `trafego`, `clima`, `tipo_entrega` e `regiao`, precisamos transformá-las em números.

Uma forma comum é usar `pd.get_dummies()`.

## Exercício 61
Crie uma versão da base com variáveis categóricas transformadas em dummies.

## Exercício 62
Mostre as primeiras linhas da base transformada.

## Exercício 63
Explique o que o `get_dummies()` fez.

## Exercício 64
Por que usamos `drop_first=True`?

## Exercício 65
Quais novas colunas foram criadas?

In [ ]:
# Exemplo inicial para a Parte H

df_modelo = pd.get_dummies(
    df,
    columns=["trafego", "clima", "tipo_entrega", "regiao"],
    drop_first=True
)

df_modelo.head()

# Parte I – Regressão linear múltipla

Agora vamos usar várias variáveis ao mesmo tempo.

## Exercício 66
Crie `X` com todas as variáveis explicativas numéricas e dummies.

Não inclua:
- `id_pedido`
- `tempo_entrega_min`
- `atrasou`

## Exercício 67
Crie `y` com `tempo_entrega_min`.

## Exercício 68
Treine um modelo de regressão linear múltipla.

## Exercício 69
Faça previsões.

## Exercício 70
Calcule MAE.

## Exercício 71
Calcule RMSE.

## Exercício 72
Calcule R².

## Exercício 73
Compare com o modelo simples.

## Exercício 74
O modelo múltiplo melhorou?

## Exercício 75
Quais variáveis parecem mais importantes?

In [ ]:
# Espaço para resolver a Parte I

# Parte J – Coeficientes do modelo múltiplo

## Exercício 76
Crie uma tabela com variáveis e coeficientes.

## Exercício 77
Ordene os coeficientes do maior para o menor.

## Exercício 78
Quais coeficientes aumentam o tempo previsto?

## Exercício 79
Quais coeficientes diminuem o tempo previsto?

## Exercício 80
Interprete o coeficiente de `distancia_km`.

## Exercício 81
Interprete o coeficiente de `tipo_entrega_expressa`.

## Exercício 82
Interprete o coeficiente de `trafego_alto`.

## Exercício 83
Interprete o coeficiente de `clima_forte_chuva`.

## Exercício 84
Os sinais dos coeficientes fazem sentido?

## Exercício 85
O que poderia indicar um coeficiente estranho?

In [ ]:
# Espaço para resolver a Parte J

# Parte K – Treino e teste

## Exercício 86
Separe os dados em treino e teste com `test_size=0.3` e `random_state=42`.

## Exercício 87
Treine o modelo nos dados de treino.

## Exercício 88
Faça previsões nos dados de teste.

## Exercício 89
Calcule MAE no teste.

## Exercício 90
Calcule RMSE no teste.

## Exercício 91
Calcule R² no teste.

## Exercício 92
Compare com as métricas do treino.

## Exercício 93
O modelo parece generalizar bem?

## Exercício 94
O que indicaria overfitting?

## Exercício 95
Quais cuidados você teria antes de colocar esse modelo em produção?

In [ ]:
# Espaço para resolver a Parte K

# Parte L – Desafio final: relatório para a empresa

A empresa quer saber se é possível prever o tempo de entrega antes do pedido sair para rota.

## Exercício 96
Treine o melhor modelo que conseguir.

## Exercício 97
Crie uma tabela com:
- tempo real
- tempo previsto
- erro
- erro absoluto

## Exercício 98
Mostre os 10 pedidos com maior erro absoluto.

## Exercício 99
Analise possíveis motivos para os maiores erros.

## Exercício 100
Escreva uma conclusão para a empresa respondendo:

1. O modelo é útil?
2. Qual o erro médio em minutos?
3. Quais variáveis mais influenciam o tempo?
4. O modelo deve ser usado sozinho para tomar decisões?
5. Que melhorias poderiam ser feitas com mais dados?

In [ ]:
# Espaço para resolver a Parte L

# Função auxiliar para avaliação de modelos

Use depois de tentar resolver manualmente.

In [ ]:
def avaliar_regressao(nome, y_real, y_pred):
    mae = mean_absolute_error(y_real, y_pred)
    mse = mean_squared_error(y_real, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_real, y_pred)

    return {
        "modelo": nome,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    }

# Fechamento

Com esta lista, você praticou um fluxo mais próximo de um problema real:

- exploração de dados;
- visualização;
- correlação;
- transformação de variáveis categóricas;
- regressão linear simples;
- regressão linear múltipla;
- treino e teste;
- avaliação de modelo;
- interpretação para tomada de decisão.